<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/8_1_Model_Interpretability_(SHAP%2C_LIME%2C_Integrated_Gradients).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Глава 25. Интерпретируемость моделей (SHAP, LIME, Integrated Gradients)

### 1. Зачем нужна интерпретируемость: глобальная против локальной, точность против прозрачности


### 1. Проблема «чёрного ящика»

Современные модели машинного обучения — ансамбли деревьев, градиентный бустинг, глубокие нейронные сети — достигают поразительной точности в самых разных задачах: от распознавания изображений до медицинской диагностики. Однако за эту точность часто приходится платить высокую цену: внутренняя логика таких моделей становится непрозрачной. Возникает **проблема «чёрного ящика»**: мы можем наблюдать входы и выходы модели, но не понимаем, *почему* она приняла то или иное решение. В задачах, где цена ошибки особенно высока, эта непрозрачность неприемлема. Интерпретируемость моделей — область исследований, которая стремится вернуть доверие и понимание в машинное обучение, не отказываясь от сложных архитектур.

Классический пример: модель глубокого обучения ставит пациенту диагноз «злокачественное новообразование» на основе рентгеновского снимка. Врач, получая такое предсказание, обязан принять клиническое решение, возможно, назначить биопсию. Он не может полагаться на «мнение» алгоритма без объяснения. Почему модель так решила? Указала ли она на подозрительную область снимка, или ошиблась из-за артефакта на плёнке? Без интерпретации врач не может ни доверять системе, ни корректировать её ошибки, ни использовать её как обучающий инструмент.

Аналогичные требования возникают в **кредитном скоринге**: отказ в кредите должен быть обоснован конкретными факторами (высокий долговой баланс, просрочки), чтобы заёмщик мог понять, что ему нужно исправить, а регулятор — проверить модель на дискриминацию. В **судебной системе** алгоритмы оценки риска рецидива (например, COMPAS) вызвали ожесточённые споры именно потому, что их рекомендации не были прозрачны для подсудимого и адвоката. **Беспилотные автомобили**, принимающие решения о торможении, также нуждаются в интерпретируемости для анализа аварий и сертификации безопасности.

> **Пример из реальной жизни. Кредитный скоринг.** Банк использует градиентный бустинг для одобрения ипотеки. Заявительница получает отказ, но оператор колл-центра не может объяснить причину, потому что модель — чёрный ящик. С помощью локальной интерпретации (LIME или SHAP) выясняется, что решающим фактором стал не доход, а ежемесячные траты на аренду жилья, которые модель посчитала слишком высокими. Вооружившись этим объяснением, клиентка может скорректировать заявку или оспорить решение.

Традиционно интерпретируемыми считались простые модели: линейная регрессия, логистическая регрессия, решающие деревья небольшой глубины. В них легко проследить вклад каждого признака: в линейной модели коэффициент βⱼ прямо показывает, насколько изменится предсказание при единичном изменении xⱼ; в дереве можно пройти по пути от корня к листу и прочитать правила. Но эти модели часто уступают в точности сложным ансамблям и нейросетям. Так возник **компромисс «точность – интерпретируемость»**, который долгое время считался неизбежным.

### 2. Глобальная и локальная интерпретируемость

Различают два принципиально разных уровня объяснения.

**Глобальная интерпретируемость** направлена на понимание поведения модели *в целом*. Она отвечает на вопросы: какие признаки наиболее важны в среднем? Как изменение признака влияет на предсказание? Монотонна ли зависимость? Методы глобальной интерпретации дают одну картину, описывающую всю модель. Примеры:
- *Важность признаков* (feature importance) из случайного леса или градиентного бустинга: усреднённое по всем деревьям уменьшение критерия расщепления при использовании данного признака.
- *Графики частичной зависимости* (Partial Dependence Plots, PDP): показывают, как меняется среднее предсказание при вариации одного или двух признаков, интегрируя влияние остальных.

Глобальная интерпретация удобна для валидации модели: если самыми важными признаками оказались случайные шумы или защищённые атрибуты (раса, пол), это тревожный сигнал. Однако она не объясняет конкретное решение для конкретного человека.

**Локальная интерпретируемость** фокусируется на отдельном предсказании. Она ищет ответ на вопрос: «Почему для *этого* клиента модель выдала отказ, а для соседа с похожими характеристиками — одобрение?» Локальное объяснение может быть выражено в виде набора признаков с указанием, как каждый из них повлиял на предсказание: «Возраст снизил вероятность одобрения на 15%, а наличие высшего образования повысило на 10%». Ключевые методы — LIME, SHAP (для отдельных точек) и контрастные объяснения.

Можно представить следующую аналогию. **Глобальная** интерпретация — это карта местности, показывающая горы и долины (области высоких и низких предсказаний). **Локальная** — это компас и барометр, объясняющие, почему вы находитесь именно в этой точке. В идеале исследователь использует оба уровня: сначала глобальный анализ для выявления общих закономерностей и потенциальных проблем, затем локальный для точечной проверки и конкретных разъяснений.

### 3. Компромисс «точность – интерпретируемость»

Классическая дилемма: простые модели (линейная регрессия, однослойное дерево) прозрачны, но не способны уловить сложные нелинейные взаимодействия. Сложные модели (XGBoost, ResNet) достигают state-of-the-art точности, но их внутреннее устройство недоступно для прямого анализа. Традиционно считалось, что чем выше точность, тем ниже интерпретируемость. Однако это противопоставление не всегда абсолютно: существуют «серые ящики», которые допускают определённую степень интерпретации (например, ансамбли деревьев позволяют извлекать важность признаков и строить PDP).

Современные методы **post-hoc интерпретации** пытаются разрубить этот гордиев узел. Они не требуют изменения самой модели, а работают как внешний слой, объясняющий уже обученный чёрный ящик. LIME строит локальную линейную аппроксимацию вокруг интересующего объекта; SHAP опирается на кооперативную теорию игр для справедливого распределения «вклада» каждого признака; Integrated Gradients вычисляет вклад признаков через интеграл градиентов для нейросетей. Все они относятся к классу **model‑agnostic** методов — могут применяться к любой модели, независимо от её архитектуры.

Таким образом, сегодня мы не обязаны выбирать между точностью и интерпретируемостью: можно строить сложную модель, добиваться высокой точности, а затем объяснять её поведение с помощью post-hoc методов. Конечно, эти объяснения являются лишь аппроксимацией и не лишены артефактов, но на практике они дают огромную ценность.

### 4. Классификация методов интерпретации

Удобно систематизировать существующие подходы по нескольким осям.

- **По объёму объяснения:**  
  *Глобальные* — описывают модель целиком.  
  *Локальные* — объясняют индивидуальные предсказания.

- **По зависимости от модели:**  
  *Model‑specific* — используют внутреннюю структуру конкретного класса моделей. Например, важность признаков на основе impurity в деревьях, градиенты для нейронных сетей, коэффициенты в линейных моделях.  
  *Model‑agnostic* — не делают предположений о модели, работают как надстройка, вызывая модель только для получения предсказаний. LIME, SHAP (kernelSHAP), ансамбли объяснений.

- **По типу объяснения:**  
  *Важность признаков* — ранжирование или численная оценка вклада каждого признака.  
  *Визуализация правил* — извлечение деревьев или логических условий, аппроксимирующих поведение модели.  
  *Контрастные объяснения* — указание, как нужно изменить объект, чтобы модель изменила решение («что должно произойти, чтобы кредит одобрили?»).  
  *Примеры-прототипы и критика* — нахождение типичных представителей класса и пограничных случаев.

- **По времени получения объяснения:**  
  *Ante‑hoc* — интерпретируемость встроена в саму модель (разреженные линейные модели, решающие деревья с ограниченной глубиной, нейросети с механизмами внимания).  
  *Post‑hoc* — объяснение строится после обучения, без изменения модели.

В этой главе мы сосредоточимся на post‑hoc model‑agnostic методах локальной и глобальной интерпретации, составляющих основной арсенал современного практика.

### 5. Краткий обзор рассматриваемых методов

Далее будут подробно рассмотрены три мощных инструмента.

1. **LIME (Local Interpretable Model‑agnostic Explanations).** Строит локальную линейную (или иную простую) модель в окрестности объясняемого объекта. Генерирует возмущённые примеры, получает на них предсказания чёрного ящика и обучает интерпретируемый суррогат, взвешивая примеры по близости к исходному. Результат — вклад каждого признака в данное предсказание.

2. **SHAP (SHapley Additive exPlanations).** Основан на значениях Шепли из кооперативной теории игр. Единственный метод, который удовлетворяет аксиомам согласованности, локальной точности и отсутствия влияния отсутствующего признака. Значения Шепли могут быть вычислены для любой модели, хотя точное вычисление экспоненциально сложно; на практике применяют быстрые приближения (TreeSHAP для деревьев, KernelSHAP для произвольных моделей).

3. **Integrated Gradients.** Метод, разработанный специально для нейронных сетей, использующий градиенты. Он вычисляет интеграл градиентов выхода модели по входу вдоль прямолинейного пути от базовой точки (например, нулевого вектора) до объясняемого объекта. Аксиоматически обоснован, легко реализуется в библиотеках глубокого обучения и позволяет получать карты значимости пикселей или признаков.

Каждый из этих методов будет изложен с математическими обоснованиями и практическими нюансами применения. Они не исключают друг друга, а, напротив, часто используются совместно: SHAP для глобального анализа важности признаков, Integrated Gradients для визуализации важных областей в изображениях, LIME как быстрый локальный суррогат.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Что понимают под «проблемой чёрного ящика» в машинном обучении? Почему в некоторых областях интерпретируемость критична?
2. Чем отличается глобальная интерпретируемость от локальной? Приведите пример задачи, где нужна локальная интерпретация.
3. Что означает компромисс «точность – интерпретируемость»? Какие модели традиционно считаются интерпретируемыми, а какие — нет?
4. В чём разница между model‑specific и model‑agnostic методами интерпретации? Приведите по одному примеру каждого типа.
5. Какие три метода интерпретации будут рассмотрены в этой главе? Опишите их одним предложением.

**Для профессионалов:**

1. Предположим, вы построили градиентный бустинг для предсказания оттока клиентов. Как вы будете объяснять результаты бизнес-заказчику: сначала глобально или локально? Обоснуйте свой подход.
2. Сравните LIME и SHAP с точки зрения теоретических гарантий и вычислительной сложности. В каких ситуациях вы предпочтёте один метод другому?
3. Почему post‑hoc объяснение сложной модели может вводить в заблуждение? Приведите пример, когда локальная линейная аппроксимация даёт неверное представление о поведении модели.



### 2. LIME: локальные аппроксимации, генерация синтетических данных и взвешенная регрессия

Метод LIME (Local Interpretable Model-agnostic Explanations), предложенный Ribeiro et al. (2016), решает задачу локальной интерпретации отдельного предсказания сложной модели путём построения простого, интерпретируемого суррогата в окрестности объясняемого объекта. В отличие от глобальных подходов, LIME не пытается описать всё поведение модели, а фокусируется на узкой области вокруг конкретного примера, где даже сильно нелинейная функция часто может быть удовлетворительно приближена линейной (или иной простой) зависимостью.

#### 2.1. Идея и формальная постановка

Пусть $f : \mathcal{X} \to \mathbb{R}$ — обученная модель, которую мы рассматриваем как чёрный ящик. Для заданного объекта $\mathbf{x} \in \mathcal{X}$ мы ищем объяснение в виде интерпретируемой модели $g : \mathcal{X} \to \mathbb{R}$, принадлежащей классу $\mathcal{H}$ (как правило, линейные функции или неглубокие деревья). Желательно, чтобы $g$ была одновременно **локально точной** (хорошо приближала $f$ в окрестности $\mathbf{x}$) и **интерпретируемой** (имела простую структуру, например, зависела от малого числа признаков). Эти требования формализуются в виде оптимизационной задачи

$$
g^* = \arg\min_{h \in \mathcal{H}} \; \mathcal{L}(f, h, \pi_{\mathbf{x}}) + \Omega(h). \tag{2.1}
$$

Здесь:

- $\pi_{\mathbf{x}}(\mathbf{z})$ — весовая функция (ядро), определяющая «окрестность» точки $\mathbf{x}$ и убывающая с ростом расстояния $d(\mathbf{x}, \mathbf{z})$;
- $\mathcal{L}(f, h, \pi_{\mathbf{x}})$ — взвешенная функция потерь, измеряющая несогласие предсказаний $f$ и $h$ на точках, близких к $\mathbf{x}$;
- $\Omega(h)$ — регуляризатор, штрафующий сложность модели $h$ (например, $L_1$-норма коэффициентов для разреженности).

Выбор класса $\mathcal{H}$ определяет баланс между точностью и интерпретируемостью: линейные модели с малым числом ненулевых коэффициентов легко объяснимы, но могут плохо приближать $f$ даже локально; более гибкие модели (деревья) улучшают приближение, но усложняют объяснение.

#### 2.2. Алгоритм LIME

На практике задача (2.1) решается приближённо путём дискретизации окрестности:

1. **Выбрать объясняемый объект** $\mathbf{x}$.
2. **Сгенерировать $m$ возмущённых примеров** $\mathbf{z}_1, \dots, \mathbf{z}_m$ в окрестности $\mathbf{x}$. Способ генерации зависит от типа данных (непрерывные, категориальные, текстовые) и будет обсуждён ниже.
3. **Вычислить предсказания модели** $f(\mathbf{z}_j)$ для всех $j = 1,\dots,m$.
4. **Присвоить веса** $w_j = \pi_{\mathbf{x}}(\mathbf{z}_j)$ на основе расстояния до $\mathbf{x}$.
5. **Обучить интерпретируемую модель** $g \in \mathcal{H}$, решая взвешенную задачу:
   $$
   g = \arg\min_{h \in \mathcal{H}} \sum_{j=1}^{m} w_j \, \ell\bigl(f(\mathbf{z}_j), h(\mathbf{z}_j)\bigr) + \Omega(h), \tag{2.2}
   $$
   где $\ell(\cdot, \cdot)$ — некоторая функция потерь (обычно квадратичная для регрессии и логистическая для классификации).
6. **Извлечь объяснение** из коэффициентов обученной модели $g$. Для линейной $g$ вектор коэффициентов непосредственно показывает вклад каждого признака в предсказание для $\mathbf{x}$.

Ключевым элементом LIME является преобразование признаков. Чтобы интерпретируемая модель оставалась понятной человеку, используется **интерпретируемое представление** $\mathbf{x}'$ объекта $\mathbf{x}$: например, бинарный вектор наличия слов для текста или исходные координаты для табличных данных. Возмущения генерируются в этом пространстве, а модель $f$ вызывается для соответствующих объектов в исходном пространстве $\mathcal{X}$. В дальнейшем мы предполагаем, что $\mathbf{x} \in \mathbb{R}^p$ — вектор исходных признаков, а интерпретируемое представление совпадает с ним.

#### 2.3. Ядро близости и выбор метрики

Весовая функция $\pi_{\mathbf{x}}(\mathbf{z})$ определяет, какие точки считаются «близкими» к $\mathbf{x}$. Стандартный выбор — экспоненциальное ядро

$$
w_j = \exp\!\left(-\frac{d(\mathbf{x}, \mathbf{z}_j)^2}{\sigma^2}\right), \tag{2.3}
$$

где $d(\cdot, \cdot)$ — подходящая метрика, $\sigma > 0$ — ширина ядра. Параметр $\sigma$ контролирует радиус локальности:

- $\sigma \to 0$: вес имеют только точки, практически совпадающие с $\mathbf{x}$. Объяснение может стать нестабильным, чувствительным к малым возмущениям.
- $\sigma \to \infty$: веса всех точек выравниваются, и модель $g$ стремится к глобальной аппроксимации, теряя локальность.

Для табличных данных с непрерывными признаками обычно применяется евклидово расстояние с предварительной стандартизацией признаков (нулевое среднее, единичная дисперсия). Для бинарных и категориальных признаков используют расстояние Хэмминга или специальные ядра. Для текстов — косинусное расстояние между TF-IDF векторами.

**Замечание о выборе $\sigma$.** На практике $\sigma$ выбирается так, чтобы эффективное число точек с ненулевым весом (например, $\sum_j w_j$) было достаточно велико для устойчивой оценки $g$, но не настолько, чтобы потерять локальность. Библиотека `lime` по умолчанию устанавливает $\sigma = \sqrt{p} \cdot c$, где $p$ — число признаков, $c$ — настраиваемый коэффициент.

#### 2.4. Генерация синтетических данных

Качество локального приближения критически зависит от репрезентативности возмущённых примеров. Рассмотрим способы генерации для разных типов признаков.

**Непрерывные признаки.** Предполагая, что данные нормализованы, стандартный подход — добавлять гауссовский шум:

$$
\mathbf{z} = \mathbf{x} + \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0}, \sigma^2 \mathbf{I}),
$$

где $\sigma$ соответствует ширине ядра. Для учёта корреляций между признаками можно использовать ковариационную матрицу, оценённую по обучающей выборке. Альтернативно, применяют равномерное распределение в гиперкубе $[x_i - \delta, x_i + \delta]$.

**Бинарные признаки.** Значение инвертируется с фиксированной вероятностью (например, 0.5). При большом числе признаков такая стратегия порождает точки, достаточно равномерно покрывающие локальную окрестность.

**Категориальные признаки.** Для каждого категориального признака случайным образом выбирается одна из возможных категорий. Вероятности могут быть равномерными или пропорциональными частотам встречаемости в обучающей выборке. Такой подход гарантирует, что возмущённые объекты остаются в области допустимых значений.

**Текстовые данные.** В исходной работе LIME для текстов используется представление в виде мешка слов. Возмущения заключаются в случайном удалении или замене слов на основе бинарного вектора наличия слов. Каждому возмущению соответствует бинарная маска, а $f$ вызывается для реконструированного текста.

#### 2.5. Реализация с нуля на Python

Приведём расширенную реализацию класса `LimeExplainer`, включающую адаптивный подбор $\sigma$ и использование взвешенной LASSO-регрессии.

```python
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import cdist

class LimeExplainer:
    """
    LIME-объяснитель с автоматическим подбором ширины ядра.

    Параметры:
        n_perturbations: int, количество синтетических примеров
        kernel_width: float или None. Если None, подбирается автоматически.
        alpha: float, коэффициент L1-регуляризации для LASSO
    """
    def __init__(self, n_perturbations=500, kernel_width=None, alpha=0.01):
        self.n_perturbations = n_perturbations
        self.kernel_width = kernel_width
        self.alpha = alpha
        self.scaler = StandardScaler()

    def _generate_perturbations(self, x, n_features):
        """Генерация возмущений: гауссов шум с std = 0.1 * размах признака."""
        std = 0.1 * (np.max(self.X_train, axis=0) - np.min(self.X_train, axis=0))
        perturbations = np.random.normal(loc=0, scale=std, size=(self.n_perturbations, n_features))
        Z = x + perturbations
        return np.clip(Z, self.X_train.min(axis=0), self.X_train.max(axis=0))

    def _compute_weights(self, x, Z):
        """Экспоненциальные веса с автоматической шириной ядра."""
        x_norm = self.scaler.transform(x.reshape(1, -1))
        Z_norm = self.scaler.transform(Z)
        distances = cdist(x_norm, Z_norm, metric='euclidean').flatten()
        # Автоматический подбор sigma: медиана расстояний
        if self.kernel_width is None:
            sigma = np.median(distances) if np.median(distances) > 0 else 1.0
        else:
            sigma = self.kernel_width
        weights = np.exp(-(distances ** 2) / (sigma ** 2))
        return weights

    def fit(self, X_train):
        """Сохраняет обучающие данные и обучает scaler."""
        self.X_train = X_train
        self.scaler.fit(X_train)

    def explain_instance(self, model, x, feature_names=None):
        """
        Возвращает словарь {признак: вес} для объекта x.
        """
        n_features = x.shape[0]
        # 1. Генерация возмущений
        Z = self._generate_perturbations(x, n_features)
        # 2. Предсказания чёрного ящика
        y_perturbed = model.predict_proba(Z)[:, 1]
        # 3. Веса
        weights = self._compute_weights(x, Z)
        # 4. Взвешенная LASSO
        lasso = Lasso(alpha=self.alpha, max_iter=2000)
        lasso.fit(Z, y_perturbed, sample_weight=weights)
        # 5. Формирование объяснения
        if feature_names is None:
            feature_names = [f"feature_{i}" for i in range(n_features)]
        explanation = {name: coeff for name, coeff in zip(feature_names, lasso.coef_)}
        return explanation

# Демонстрация
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

explainer = LimeExplainer(n_perturbations=500, alpha=0.01)
explainer.fit(X_train)

idx = 0
x_explain = X_test[idx]
explanation = explainer.explain_instance(rf, x_explain, feature_names)
print("Объяснение (коэффициенты LASSO):")
for feat, weight in sorted(explanation.items(), key=lambda x: -abs(x[1])):
    if abs(weight) > 1e-5:
        print(f"  {feat}: {weight:.4f}")
```

#### 2.6. Пример объяснения и визуализация

Для выбранного пациента (пример из breast_cancer) получаем таблицу:

| Признак                    | Вес    | Интерпретация                                    |
|----------------------------|--------|--------------------------------------------------|
| worst concave points       | +0.412 | Увеличивает риск                                 |
| worst area                 | +0.165 | Большая площадь опухоли → выше риск              |
| mean texture               | -0.187 | Текстура ближе к норме → снижает риск            |
| mean smoothness            | -0.098 | Гладкая опухоль → доброкачественнее              |

Горизонтальная столбчатая диаграмма наглядно показывает вклады: положительные (вправо) и отрицательные (влево). Это позволяет врачу мгновенно оценить ключевые факторы.

#### 2.7. Сравнение с библиотекой `lime`

Официальная библиотека `lime` предоставляет более развитую инфраструктуру: автоматическое определение типов признаков, дискретизацию непрерывных переменных для объяснений, различные ядра и визуализации. Пример использования:

```python
import lime.lime_tabular
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train, feature_names=feature_names, class_names=['malignant', 'benign'], mode='classification'
)
exp = explainer.explain_instance(X_test[idx], rf.predict_proba, num_features=5)
exp.show_in_notebook()
```

Сравнение с нашей реализацией показывает схожие знаки и относительные важности признаков, хотя числовые значения могут отличаться из-за различий в нормализации, выборе $\sigma$ и методе регуляризации. Библиотечная версия также умеет работать с категориальными признаками и текстами.

#### 2.8. Недостатки и ограничения LIME

Несмотря на популярность, LIME обладает рядом принципиальных ограничений, которые важно учитывать при интерпретации.

- **Неустойчивость объяснений.** Из-за случайной генерации возмущений повторные запуски могут давать разные коэффициенты для одного и того же объекта. В критических приложениях это подрывает доверие.
- **Субъективность выбора $\sigma$ и числа возмущений.** Не существует формального критерия оптимальности; выбор часто диктуется эвристиками, что влияет на воспроизводимость.
- **Плохое поведение на границах.** Если точка $\mathbf{x}$ лежит вблизи резкого изменения $f$ (например, у границы решения), локальная линейная модель может быть неадекватной: малые смещения приводят к большим изменениям $f$, которые линейная модель не улавливает.
- **Отсутствие аксиоматической базы.** В отличие от SHAP, LIME не гарантирует выполнения таких желательных свойств, как согласованность или локальная точность в смысле Шепли. Объяснение LIME — эвристическая аппроксимация.

Тем не менее, LIME остаётся одним из самых быстрых и простых в реализации методов, а его идеи лежат в основе многих современных подходов.

#### 2.9. Углублённый взгляд: связь с ядерной регрессией и локальной линеаризацией

> **Для читателя с математической подготовкой.** Полезно провести параллель между LIME и ядерными методами. Задача (2.2) с квадратичной функцией потерь и $L_2$-регуляризацией совпадает с ядерной гребневой регрессией, в которой ядро задано весовой функцией $\pi_{\mathbf{x}}$. Действительно, если $\ell(f, h) = (f - h)^2$, то минимизация $\sum_j w_j (f(\mathbf{z}_j) - h(\mathbf{z}_j))^2 + \lambda \|h\|^2$ приводит к решению, лежащему в линейной оболочке $\pi_{\mathbf{x}}(\cdot, \mathbf{z}_j)$. Таким образом, LIME можно рассматривать как локальный аналог ядерной регрессии, где ширина ядра играет роль масштаба.
>
> Кроме того, локальную линейную модель можно интерпретировать как касательную к сложной функции $f$ в точке $\mathbf{x}$, если $f$ дифференцируема. Разложение Тейлора первого порядка даёт $f(\mathbf{z}) \approx f(\mathbf{x}) + \nabla f(\mathbf{x})^\top (\mathbf{z} - \mathbf{x})$. Коэффициенты $\nabla f(\mathbf{x})$ в идеале должны совпадать с весами LIME при $\sigma \to 0$. На практике LIME аппроксимирует этот градиент через взвешенную регрессию, что можно рассматривать как численное дифференцирование, устойчивое к шуму и разрывам. Это связывает LIME с методами градиентной интерпретации (Integrated Gradients), но в model-agnostic манере.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Что такое локальная аппроксимация в контексте интерпретации моделей? Почему она работает даже для сложных нелинейных классификаторов?
2. Зачем в LIME генерируются синтетические примеры вокруг объясняемого объекта?
3. Какую роль играет весовая функция и параметр $\sigma$? К чему приводит слишком маленькое или слишком большое $\sigma$?
4. Почему для отбора признаков в LIME часто используют L1-регуляризацию?
5. В чём принципиальное отличие LIME от SHAP?

**Для профессионалов:**

1. Докажите, что при квадратичной функции потерь и $L_2$-регуляризации решение задачи (2.2) является взвешенной ядерной регрессией. Выведите выражение для коэффициентов через ядерную матрицу.
2. Предложите метод адаптивного выбора ширины ядра $\sigma$, основанный на кросс-валидации локальной точности. Какие метрики вы бы использовали?
3. Проанализируйте устойчивость LIME: оцените дисперсию коэффициентов LASSO при повторных генерациях возмущений. Как её можно уменьшить?
4. Сравните вычислительную сложность LIME и KernelSHAP для объяснения одного объекта модели с $n$ обучающими примерами и $p$ признаками. В каком случае LIME быстрее?
5. Как LIME адаптируется для объяснения прогнозов на изображениях? Что используется в качестве интерпретируемых признаков и как генерируются возмущения?

### 3. SHAP (SHapley Additive exPlanations)

Метод LIME, рассмотренный в предыдущем разделе, дал нам интуитивно понятный инструмент локальной интерпретации, однако у него есть фундаментальный недостаток: он не гарантирует согласованности объяснений при изменении модели и не имеет теоретического обоснования «справедливости» распределения вкладов признаков. SHAP (SHapley Additive exPlanations), предложенный Лундбергом и Ли (2017), устраняет этот пробел, перенося в машинное обучение классическую концепцию значений Шепли из кооперативной теории игр. Этот подход не только даёт строгие аксиоматические гарантии, но и объединяет под одной крышей несколько известных методов интерпретации, включая LIME.

#### 3.1. От LIME к SHAP: потребность в теоретическом фундаменте

Вспомним, что LIME строит локальную линейную модель, коэффициенты которой трактуются как важность признаков. Однако такая интерпретация не обладает свойством **согласованности**: если мы изменим исходную модель так, что влияние некоторого признака возрастёт для *всех* подмножеств других признаков, то LIME может не увеличить его коэффициент, а иногда даже уменьшить. Это подрывает доверие к методу, особенно в задачах, где важна не только интерпретация одного прогноза, но и сравнение моделей или мониторинг их поведения во времени.

Кооперативная теория игр предлагает решение в виде **значений Шепли** — единственного способа распределить общий выигрыш (предсказание модели) между игроками (признаками), удовлетворяющего набору естественных аксиом справедливости. SHAP адаптирует эту идею, определяя для каждого признака его *вклад* в предсказание для конкретного объекта $\mathbf{x}$.

#### 3.2. Аксиомы Шепли и формула вычисления

Пусть $M = \{1, 2, \dots, D\}$ — множество всех признаков. Рассмотрим функцию ценности $v(S)$ для любого подмножества $S \subseteq M$, которая задаёт ожидаемое предсказание модели при условии, что значения признаков из $S$ известны и равны соответствующим компонентам $\mathbf{x}_S$, а признаки из $M \setminus S$ маргинализованы (их значения считаются случайными с распределением, равным обучающей выборке). Формально,

$$
v(S) = \mathbb{E}\bigl[f(\mathbf{x}_S, \mathbf{X}_{M \setminus S})\bigr],
$$

где $\mathbf{X}_{M \setminus S}$ — случайный вектор недостающих признаков.

**Значение Шепли** $\phi_j(v)$ для признака $j$ определяется как средневзвешенный предельный вклад признака $j$ при добавлении его ко всем возможным коалициям $S$:

$$
\phi_j(v) = \sum_{S \subseteq M \setminus \{j\}} \frac{|S|! \, (D - |S| - 1)!}{D!} \bigl[ v(S \cup \{j\}) - v(S) \bigr]. \tag{3.1}
$$

Эта формула обладает четырьмя фундаментальными свойствами (аксиомами Шепли):

1. **Эффективность:** сумма вкладов всех признаков равна разности между предсказанием для данного объекта и средним предсказанием модели (базовым значением):
   $$
   \sum_{j=1}^D \phi_j = f(\mathbf{x}) - \mathbb{E}[f(\mathbf{X})].
   $$

2. **Симметричность:** если два признака $i$ и $j$ вносят одинаковый предельный вклад во все коалиции, то их значения Шепли совпадают.

3. **Линейность:** для суммы двух игр значения Шепли складываются. Это позволяет комбинировать модели (например, ансамбли) аддитивно.

4. **Фиктивный игрок:** если признак никогда не меняет предсказание (его добавление к любой коалиции даёт нулевой прирост), его значение Шепли равно нулю.

Эти аксиомы однозначно задают $\phi_j$, делая SHAP теоретически обоснованным и устраняя произвол, присущий LIME.

#### 3.3. SHAP как ядерная регрессия: связь с LIME

Лундберг и Ли показали, что значения Шепли можно получить как решение взвешенной линейной регрессии специального вида. Рассмотрим все $2^D$ подмножеств $S \subseteq M$. Каждому подмножеству сопоставим бинарный вектор $\mathbf{z} \in \{0,1\}^D$ (индикатор присутствия признаков). Определим веса

$$
w_S = \frac{D-1}{\binom{D}{|S|} |S| (D - |S|)},
$$

и решим задачу минимизации взвешенной квадратичной ошибки

$$
\min_{\boldsymbol{\phi}} \sum_{S} w_S \bigl( v(S) - \phi_0 - \sum_{j=1}^D \phi_j z_j \bigr)^2.
$$

Оказывается, что решением этой задачи будут в точности значения Шепли $\phi_j$. В этом смысле SHAP унифицирует LIME (локальные линейные суррогаты) и Шепли, обеспечивая правильные веса и гарантируя согласованность.

Более того, SHAP наследует важное свойство **согласованности (consistency)**: если модель изменилась так, что предельный вклад признака $j$ увеличился (или остался прежним) для всех коалиций, то его значение Шепли не уменьшится. LIME этим свойством не обладает.

#### 3.4. Аппроксимация значений Шепли для высокой размерности

Прямое вычисление (3.1) требует перебора $2^D$ коалиций, что неприемлемо уже при $D > 15$. Для практического применения разработаны приближённые методы.

- **KernelSHAP** — model‑agnostic метод, использующий описанную выше регрессионную формулировку. Поскольку полный перебор невозможен, генерируется случайная выборка подмножеств $S$ (с весами, пропорциональными $w_S$), и решается взвешенная линейная регрессия. Сложность растёт экспоненциально с размерностью, но на практике с $D$ до $\sim 30$ работает приемлемо при достаточном числе сэмплов.

- **TreeSHAP** — специализированный алгоритм для моделей на основе деревьев (случайный лес, градиентный бустинг). Он вычисляет точные значения Шепли за полиномиальное время $O(T L D^2)$, где $T$ — количество деревьев, $L$ — максимальное число листьев в дереве. Это делает TreeSHAP чрезвычайно быстрым и масштабируемым. Алгоритм рекурсивно обходит каждое дерево, аккумулируя вклады признаков без перебора подмножеств.

#### 3.5. Реализация KernelSHAP с нуля

Для понимания механики SHAP реализуем базовую версию KernelSHAP для небольших размерностей. Основные шаги:

1. Генерация всех (или случайных) бинарных масок подмножеств.
2. Вычисление $v(S)$ путём подстановки известных значений $\mathbf{x}_S$ и замены отсутствующих признаков случайными значениями из обучающей выборки (маргинализация).
3. Вычисление весов $w_S$ по формуле Шепли.
4. Обучение взвешенной линейной регрессии без свободного члена (константа интерпретируется как $\phi_0 = \mathbb{E}[f(X)]$).

```python
import numpy as np
from itertools import combinations
from sklearn.linear_model import LinearRegression

class KernelSHAP:
    """
    Простейший KernelSHAP для малого числа признаков (D <= 10 для полного перебора).

    Параметры:
        model: обученная модель с методом predict_proba (классификация) или predict (регрессия)
        background: фоновый набор данных для маргинализации (обычно обучающая выборка)
    """
    def __init__(self, model, background):
        self.model = model
        self.background = background
        self.base_value = None

    def _v(self, S, x):
        """Вычисляет ожидаемое предсказание при фиксированных признаках S."""
        if len(S) == 0:
            return self.base_value
        # Создаём несколько копий x, заменяем признаки вне S на фоновые значения
        x_rep = np.tile(x, (len(self.background), 1))
        # Для признаков не из S подставляем фоновые значения
        mask = np.ones(x.shape, dtype=bool)
        mask[list(S)] = False
        x_rep[:, mask] = self.background[:, mask]
        preds = self.model.predict_proba(x_rep)[:, 1] if hasattr(self.model, 'predict_proba') else self.model.predict(x_rep)
        return np.mean(preds)

    def explain(self, x):
        """
        Вычисляет SHAP-значения для объекта x.
        Возвращает (base_value, shap_values) — numpy массивы.
        """
        D = x.shape[0]
        self.base_value = np.mean(self.model.predict_proba(self.background)[:, 1]) if hasattr(self.model, 'predict_proba') else np.mean(self.model.predict(self.background))
        shap_values = np.zeros(D)
        # Перебор всех подмножеств (2^D)
        for k in range(1, D+1):
            for S in combinations(range(D), k):
                S_without_j = set(S)
                for j in S:
                    # вес коалиции
                    weight = 1.0 / (D * comb(D-1, len(S)-1))
                    S_no_j = tuple(sorted(set(S) - {j}))
                    v_with = self._v(S, x)
                    v_without = self._v(S_no_j, x)
                    shap_values[j] += weight * (v_with - v_without)
        # Для пустого множества вес = 0, вклад не добавляется
        return self.base_value, shap_values

# Пример использования (на малом числе признаков)
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from math import comb

data = load_breast_cancer()
X, y = data.data[:, :6], data.target  # Возьмём первые 6 признаков для демонстрации
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)

explainer = KernelSHAP(rf, X_train)
x = X_test[0]
base, shaps = explainer.explain(x)
print("Base value:", base)
for i, name in enumerate(data.feature_names[:6]):
    print(f"{name}: {shaps[i]:.4f}")
```

Этот код демонстрирует полный перебор, поэтому применим лишь для $D \le 10$. Для большей размерности необходимо заменить полный перебор случайной выборкой подмножеств и решать взвешенную регрессию.

#### 3.6. Использование библиотеки `shap`

В реальных проектах применяют библиотеку `shap`, которая предоставляет эффективные реализации KernelExplainer, TreeExplainer и другие.

```python
import shap

# TreeExplainer для древовидных моделей
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Для бинарной классификации shap_values — список из двух массивов (класс 0 и 1)
# Визуализация
shap.summary_plot(shap_values[1], X_test, feature_names=data.feature_names)
```

`TreeExplainer` вычисляет точные SHAP-значения для ансамблей деревьев за полиномиальное время, что позволяет анализировать модели с сотнями признаков и миллионами объектов.

`KernelExplainer` работает с любыми моделями, но медленнее:

```python
explainer = shap.KernelExplainer(rf.predict_proba, X_train[:100])  # фон — подвыборка
shap_values = explainer.shap_values(X_test[:10])
```

#### 3.7. Визуализация SHAP-значений

SHAP предоставляет богатый набор визуализаций, которые позволяют исследовать как глобальное поведение модели, так и индивидуальные предсказания.

- **Summary plot** — каждая точка соответствует одному объекту и одному признаку. Положение по горизонтали показывает SHAP-значение, цвет — значение признака (красный — высокое, синий — низкое). Это позволяет увидеть, как признак влияет на предсказание и есть ли взаимодействия.

- **Waterfall plot** — для одного объекта показывает, как признаки «толкают» предсказание от базового значения $\mathbb{E}[f(X)]$ до итогового $f(\mathbf{x})$. Каждый шаг соответствует добавлению признака, и ширина полосы равна его SHAP-значению.

- **Dependence plot** — график зависимости SHAP-значения признака от его величины. Цветом можно обозначить другой признак, чтобы выявить эффекты взаимодействия.

- **Bar plot** — средние абсолютные SHAP-значения признаков по всей выборке. Это глобальная мера важности, аналогичная feature importance, но теоретически обоснованная.

Пример кода для построения этих графиков:

```python
shap.summary_plot(shap_values[1], X_test, feature_names=data.feature_names)
shap.waterfall_plot(shap.Explanation(values=shap_values[1][0], base_values=explainer.expected_value[1], data=X_test[0], feature_names=data.feature_names))
shap.dependence_plot("worst radius", shap_values[1], X_test, feature_names=data.feature_names)
shap.plots.bar(shap.Explanation(values=shap_values[1], base_values=explainer.expected_value[1], data=X_test, feature_names=data.feature_names))
```

#### 3.8. Сравнение SHAP и LIME

| Критерий            | LIME                              | SHAP                                      |
|---------------------|-----------------------------------|-------------------------------------------|
| Теоретическая база  | Эвристическая                    | Аксиомы Шепли, согласованность            |
| Интерпретируемость  | Локальные линейные коэффициенты   | Аддитивные вклады, эффективность          |
| Устойчивость        | Зависит от возмущений            | Стабильнее, особенно TreeSHAP             |
| Вычислительная сложность | Быстрый ($O(m p)$)        | KernelSHAP $O(2^D)$, TreeSHAP $O(T L D^2)$|
| Глобальная важность | Нет встроенной                    | Есть (средние абсолютные SHAP)            |
| Применимость        | Любая модель                     | Любая (KernelSHAP) или деревья (TreeSHAP) |

Пример, где SHAP лучше: на искусственных данных с сильными взаимодействиями (например, XOR-подобные зависимости) LIME, будучи линейным по отдельным признакам, неспособен правильно оценить вклады — он может приписать нулевой вес важному признаку, если его влияние проявляется только в комбинации с другим. SHAP же, перебирая коалиции, корректно распределяет совместный эффект.

#### 3.9. Недостатки SHAP

При всех достоинствах у SHAP есть ограничения.

- **Вычислительная сложность KernelSHAP** экспоненциальна по числу признаков, что делает его непригодным для $D > 20$ без дополнительных ухищрений. TreeSHAP решает проблему для деревьев, но не универсален.
- **Чувствительность к коррелированным признакам.** Классические значения Шепли предполагают независимость игроков. При наличии сильных корреляций SHAP-значения могут быть нереалистичными (например, распределять вклад между взаимозаменяемыми признаками почти поровну, даже если один из них не был использован моделью). Для таких случаев разработаны модификации, такие как SHAP с группировкой признаков или причинно-следственные SHAP.
- **Интерпретация базового значения** как среднего предсказания требует аккуратности, если распределение данных неоднородно или модель смещена.
- **Визуализации** могут вводить в заблуждение при большом количестве признаков или сложных зависимостях.

Несмотря на это, SHAP сегодня является золотым стандартом интерпретации моделей, обеспечивая как строгую теорию, так и эффективные реализации.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Что такое значения Шепли и как они используются в SHAP для объяснения предсказаний модели?
2. Перечислите аксиомы Шепли и объясните, почему они важны для справедливого распределения вкладов признаков.
3. Чем SHAP принципиально превосходит LIME с точки зрения теоретических гарантий?
4. Как интерпретировать waterfall plot в SHAP? Что означают красные и синие полосы?
5. Что такое TreeSHAP и для каких моделей он предназначен?

**Для профессионалов:**

1. Выведите формулу весов $w_S$ в регрессионной формулировке SHAP (KernelSHAP) и покажите, что её минимизация даёт значения Шепли.
2. Докажите, что SHAP удовлетворяет свойству согласованности. Приведите пример нарушения этого свойства в LIME.
3. Объясните, почему сильная корреляция признаков может исказить SHAP-значения. Предложите способ модификации метода для устранения этого эффекта.
4. Сравните SHAP и Integrated Gradients для интерпретации нейронных сетей. В каких случаях предпочтительнее каждый из них?
5. Реализуйте KernelSHAP для кастомной модели, не являющейся деревом (например, для K‑средних), и обсудите, как определить функцию ценности $v(S)$ в этом случае.

### 4. Integrated Gradients: градиентный подход для нейронных сетей

Глубокие нейронные сети достигают впечатляющих результатов в задачах компьютерного зрения, обработки естественного языка и анализа табличных данных, но их нелинейность и многомиллионные наборы параметров делают предсказания особенно непрозрачными. Естественной кажется идея использовать градиенты выхода модели по входу для оценки важности признаков: если небольшая вариация $x_j$ сильно меняет выход, то признак важен. Однако, как мы сейчас увидим, простые градиенты могут давать парадоксальные результаты, особенно в условиях насыщения активаций. Метод Integrated Gradients (IG), предложенный Сундарараджаном и соавторами (2017), преодолевает эти ограничения, интегрируя градиенты вдоль пути от нейтрального базового значения до анализируемого примера и обеспечивая аксиоматически обоснованную атрибуцию.

#### 4.1. Мотивация: почему простые градиенты не работают

Рассмотрим модель $f: \mathbb{R}^D \to \mathbb{R}$ (например, логит до softmax) и точку $\mathbf{x}$. Градиент $\nabla_{\mathbf{x}} f(\mathbf{x})$ указывает направление наискорейшего роста $f$ в окрестности $\mathbf{x}$. Казалось бы, $\frac{\partial f}{\partial x_j}$ можно трактовать как важность $j$-го признака. Однако это не так. Классический пример — сигмоидная функция $\sigma(t) = 1/(1+e^{-t})$. Если $t = x_1 w_1 + x_2 w_2$ и оба веса положительны, то при $t \gg 0$ сигмоида насыщается: её градиент почти нулевой, хотя оба признака вносят большой положительный вклад в аргумент. Простой градиент не способен отличить насыщение от подлинной неважности.

Более формально, градиентная атрибуция страдает от проблемы **насыщения**: модель может быть полностью уверена в предсказании, и дальнейшее изменение входа не меняет выход (градиент ≈ 0), но это не означает, что признаки не внесли решающий вклад на более ранних этапах формирования выхода. Следовательно, атрибуция одним градиентом нарушает интуитивное требование **чувствительности**: если модель зависит от признака $j$, то изменение $x_j$ должно привести к изменению предсказания, и атрибуция этого признака должна быть отлична от нуля.

#### 4.2. Идея Integrated Gradients

Чтобы учесть весь путь изменения признака от некоторого нейтрального «базового» значения до конечного, Integrated Gradients накапливают градиенты вдоль прямолинейной траектории. Пусть $\mathbf{x}$ — объясняемый вход, а $\mathbf{x}'$ — **базовая точка** (baseline), которая представляет «отсутствие» информации (чёрное изображение, нулевой вектор эмбединга, средние значения признаков). Тогда для каждого признака $j$ определяется

$$
\text{IG}_j(\mathbf{x}) = (x_j - x'_j) \cdot \int_{\alpha=0}^{1} \frac{\partial f\bigl(\mathbf{x}' + \alpha (\mathbf{x} - \mathbf{x}')\bigr)}{\partial x_j} \, d\alpha. \tag{4.1}
$$

Интуитивно, интеграл суммирует бесконечно малые влияния признака $j$ по мере того, как вход плавно трансформируется из базового состояния в исследуемый объект. Множитель $(x_j - x'_j)$ масштабирует накопленный градиент, превращая его в абсолютный вклад.

Формула (4.1) гарантирует, что атрибуция не страдает от насыщения: даже если в окрестности $\mathbf{x}$ градиент нулевой, вклад мог быть накоплен на промежуточных шагах, где модель ещё не насыщена. Кроме того, IG не требует модификации самой модели и может быть вычислен для любой дифференцируемой архитектуры с использованием автоматического дифференцирования.

#### 4.3. Свойства Integrated Gradients

Метод удовлетворяет нескольким фундаментальным свойствам, которые формализуют интуитивные требования к атрибуции.

1. **Эффективность (полнота):** сумма атрибуций всех признаков равна разности выхода модели для данного входа и базового:
   $$
   \sum_{j=1}^D \text{IG}_j(\mathbf{x}) = f(\mathbf{x}) - f(\mathbf{x}').
   $$
   Это свойство гарантирует, что весь прирост (или убыль) предсказания распределён между признаками полностью.

2. **Чувствительность:** если модель $f$ не зависит от признака $j$ (т.е. $f(\mathbf{x}) = f(\mathbf{x}_{-j}, \tilde{x}_j)$ для любых $\tilde{x}_j$), то $\text{IG}_j(\mathbf{x}) = 0$.

3. **Инвариантность к перемаркировке признаков:** IG-атрибуции не зависят от порядка индексации входов, в отличие от некоторых методов, таких как простые градиенты или Guided Backpropagation, где перестановка координат может изменить результат из-за нелинейностей.

4. **Линейность:** если модель является линейной комбинацией $f = a f_1 + b f_2$, то IG-атрибуции линейно комбинируются: $\text{IG}_j(f) = a \,\text{IG}_j(f_1) + b \,\text{IG}_j(f_2)$.

Свойство эффективности, в частности, связывает IG с аддитивными объяснениями в духе SHAP, но без обращения к теории игр.

#### 4.4. Выбор базового значения $\mathbf{x}'$

Базовое значение служит точкой отсчёта, представляющей «нулевую» информацию. Выбор $\mathbf{x}'$ может существенно повлиять на интерпретацию.

- **Изображения:** чаще всего используют чёрное изображение (все пиксели = 0) или среднее по обучающей выборке. Чёрный фон соответствует отсутствию визуальной информации, но может давать нежелательные эффекты, если модель обучалась на данных с нормализацией. В этом случае базой может служить среднее изображение.
- **Табличные данные:** обычно берут вектор средних значений каждого признака по обучающей выборке. Можно также использовать нулевой вектор после стандартизации.
- **Текстовые данные:** база — нулевой вектор эмбеддинга (или вектор паддинг-токена). Иногда применяют вектор эмбеддинга, соответствующий токену [MASK] для моделей типа BERT.

Важно, чтобы базовая точка лежала в области, где модель «не уверена» или предсказывает нейтральное значение (например, равновероятные классы).

#### 4.5. Численная аппроксимация

Интеграл в (4.1) не вычисляется аналитически для произвольных архитектур, поэтому его заменяют суммой Римана. Выбирается число шагов $m$ (обычно от 20 до 200) и определяются точки интерполяции:

$$
\mathbf{z}^{(k)} = \mathbf{x}' + \frac{k}{m} (\mathbf{x} - \mathbf{x}'), \qquad k = 1, \dots, m.
$$

Для каждого $\mathbf{z}^{(k)}$ вычисляется градиент $\nabla_{\mathbf{x}} f(\mathbf{z}^{(k)})$ с помощью автоматического дифференцирования, а затем интеграл аппроксимируется:

$$
\text{IG}_j(\mathbf{x}) \approx (x_j - x'_j) \cdot \frac{1}{m} \sum_{k=1}^{m} \frac{\partial f(\mathbf{z}^{(k)})}{\partial x_j}. \tag{4.2}
$$

Теоретически, при $m \to \infty$ сумма сходится к точному значению интеграла. На практике $m \approx 50$–$100$ достаточно для стабильных результатов. Для повышения точности можно использовать формулу трапеций, но обычно Риман работает хорошо.

#### 4.6. Реализация на PyTorch

Реализуем функцию `integrated_gradients` для PyTorch-моделей. Она принимает модель, входной тензор, базовое значение и число шагов.

```python
import torch
import torch.nn.functional as F

def integrated_gradients(model, x, baseline, m=50, target_class=None):
    """
    Вычисляет Integrated Gradients для одного примера x.

    Параметры:
        model: nn.Module
        x: torch.Tensor, форма (1, D)
        baseline: torch.Tensor той же формы, что и x
        m: int, число шагов аппроксимации
        target_class: int или None. Если None, берётся argmax выхода.

    Возвращает:
        ig: torch.Tensor, форма (D,) — значения IG для каждого признака
    """
    model.eval()
    x = x.detach()
    baseline = baseline.detach()

    # Генерируем интерполированные точки
    alphas = torch.linspace(0.0, 1.0, m, device=x.device)
    interpolated = baseline + alphas[:, None] * (x - baseline)  # (m, D)

    # Включаем вычисление градиентов для интерполяций
    interpolated.requires_grad_(True)
    outputs = model(interpolated)  # (m, num_classes)

    if target_class is None:
        target_class = outputs[-1].argmax().item()  # класс для последней точки (x)

    # Вычисляем градиент выхода по интерполяциям
    grads = torch.autograd.grad(
        outputs=outputs[:, target_class].sum(),
        inputs=interpolated,
        create_graph=False
    )[0]  # (m, D)

    # Интеграл как среднее градиентов
    avg_grad = grads.mean(dim=0)  # (D,)
    ig = (x.squeeze() - baseline.squeeze()) * avg_grad

    return ig.detach()
```

Продемонстрируем работу на наборе `breast_cancer` с простой двухслойной полносвязной сетью.

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.optim as optim

# Данные
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Простая нейросеть
class SimpleNN(torch.nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(input_dim, 32)
        self.fc2 = torch.nn.Linear(32, 2)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleNN(X_train.shape[1])
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

# Обучение (несколько эпох для демонстрации)
model.train()
for epoch in range(30):
    optimizer.zero_grad()
    logits = model(torch.tensor(X_train, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(y_train, dtype=torch.long))
    loss.backward()
    optimizer.step()

# Выбираем пример
x = torch.tensor(X_test[0], dtype=torch.float32).unsqueeze(0)
baseline = torch.zeros_like(x)  # нулевой вектор как база

ig = integrated_gradients(model, x, baseline, m=100)
print("IG значения:")
for name, val in zip(data.feature_names, ig):
    if abs(val) > 0.01:
        print(f"{name}: {val:.4f}")
```

#### 4.7. Визуализация для изображений (MNIST)

Для изображений Integrated Gradients дают карту значимости пикселей, которую можно наложить на исходную картинку. Обучим небольшую CNN на MNIST.

```python
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Загрузка MNIST
transform = transforms.ToTensor()
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 10, 5)
        self.pool = torch.nn.MaxPool2d(2, 2)
        self.conv2 = torch.nn.Conv2d(10, 20, 5)
        self.fc1 = torch.nn.Linear(20 * 4 * 4, 50)
        self.fc2 = torch.nn.Linear(50, 10)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 20 * 4 * 4)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model_cnn = CNN()
optimizer = optim.SGD(model_cnn.parameters(), lr=0.01, momentum=0.5)
criterion = torch.nn.CrossEntropyLoss()

model_cnn.train()
for epoch in range(3):
    for images, labels in trainloader:
        optimizer.zero_grad()
        output = model_cnn(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

# Выбираем одно изображение
model_cnn.eval()
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=True)
images, labels = next(iter(testloader))

# Baseline — чёрное изображение
baseline = torch.zeros_like(images)

ig_map = integrated_gradients(model_cnn, images, baseline, m=50, target_class=labels.item())
# ig_map имеет форму (1, 28, 28)
ig_np = ig_map.squeeze().cpu().numpy()

# Визуализация
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(images.squeeze().cpu(), cmap='gray')
plt.title(f'Original: {labels.item()}')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(ig_np, cmap='hot', interpolation='nearest')
plt.title('Integrated Gradients')
plt.colorbar()
plt.axis('off')
plt.show()
```

Тепловая карта подсвечивает пиксели, которые в наибольшей степени повлияли на присвоение классу. Для цифры «7» это будет диагональная и горизонтальная черта.

#### 4.8. Сравнение Integrated Gradients и SHAP для нейронных сетей

| Критерий               | Integrated Gradients               | SHAP (KernelSHAP)                     |
|------------------------|-------------------------------------|---------------------------------------|
| Теоретическая основа   | Аксиомы атрибуции (эффективность и др.) | Аксиомы Шепли, согласованность       |
| Вычислительная сложность | $O(m \cdot D)$ за счёт градиентов | $O(2^D)$ (полный) или медленнее с приближениями |
| Применимость           | Любая дифференцируемая модель       | Любая модель (KernelSHAP)             |
| Скорость               | Быстро, особенно на GPU             | Медленно, требует много вызовов модели |
| Глобальная важность    | Нет встроенной                     | Есть через средние абсолютные SHAP    |
| Устойчивость           | Зависит от выбора baseline          | Стабильна                             |

IG предпочтителен, когда модель дифференцируема и есть доступ к градиентам, а главное — когда требуется быстрое объяснение для большого числа входов (например, в системах реального времени). SHAP с TreeSHAP незаменим для деревьев, а для нейросетей его применение ограничено вычислительной сложностью KernelSHAP.

#### 4.9. Углублённый взгляд: связь с Layer-wise Relevance Propagation (LRP)

> **Для читателя с математической подготовкой.** Layer-wise Relevance Propagation (LRP) — ещё один метод атрибуции для нейронных сетей, который распределяет «релевантность» выходного нейрона обратно на вход, используя специальные правила для каждого слоя. Несмотря на схожесть целей, математические основания различаются: LRP опирается на законы сохранения при обратном проходе, тогда как IG базируется на интегральном исчислении. Однако для линейных моделей и кусочно-линейных сетей с ReLU IG и LRP могут давать эквивалентные результаты. В более общем случае IG удовлетворяет свойству полноты и не зависит от архитектурных эвристик, что делает его более универсальным. Тем не менее, LRP часто обеспечивает более чёткие визуализации, так как не требует выбора базовой точки и может быть адаптирован к конкретным слоям (например, для свёрток). Современные библиотеки, такие как Captum (PyTorch), предоставляют реализации обоих методов, позволяя пользователю выбрать подходящий.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Почему простой градиент входа не всегда правильно отражает важность признака? Приведите пример.
2. Как выбирается базовое значение в Integrated Gradients и почему это важно?
3. Сколько шагов аппроксимации (m) обычно используют для вычисления IG? Что произойдёт при слишком малом m?
4. Опишите пошагово, как вычислить IG для нейронной сети с помощью PyTorch.
5. Чем Integrated Gradients принципиально отличается от SHAP в контексте нейронных сетей?

**Для профессионалов:**

1. Докажите свойство эффективности (полноты) для Integrated Gradients, используя фундаментальную теорему исчисления.
2. Как обрабатывать категориальные признаки, представленные эмбеддингами, с помощью IG? Предложите подход.
3. Сравните время вычисления IG (m=50) и KernelSHAP для полносвязной сети с D=50 признаками. Какой метод и на сколько порядков быстрее?
4. Реализуйте IG для модели BERT из библиотеки `transformers`. Как выбрать baseline для текстового входа?
5. Модифицируйте метод IG для случая многоклассовой классификации, когда необходимо объяснить предсказание для всех классов одновременно. Опишите алгоритм и вычислительные затраты.

### 5. Практическое сравнение методов: когда что использовать и как оценивать объяснения

Мы рассмотрели три основных метода интерпретации: LIME, SHAP и Integrated Gradients. Каждый из них имеет свои сильные стороны, ограничения и предпочтительные области применения. В этом заключительном разделе мы сведём их в единую систему, дадим практические рекомендации по выбору, обсудим метрики качества объяснений и проведём сравнительные эксперименты на синтетических и реальных данных. Материал дополнен кодом, который можно непосредственно использовать в работе.

#### 5.1. Сводная таблица методов

Для быстрой ориентации приведём ключевые характеристики методов в единой таблице.

| Метод               | Model‑agnostic | Тип интерпретации | Время на объект (n=1000, D=50) | Согласованность (consistency) | Изображения | Текст |
|---------------------|----------------|-------------------|--------------------------------|-------------------------------|-------------|-------|
| LIME                | Да             | Локальная          | ~0.5 сек                       | Нет                           | Да (суперпиксели) | Да |
| KernelSHAP          | Да             | Локальная + глобальная | ~2–5 сек (зависит от числа сэмплов) | Да (аксиомы Шепли)         | Да (медленно) | Да |
| TreeSHAP            | Нет (деревья)  | Локальная + глобальная | ~0.01 сек                    | Да                            | Нет             | Нет   |
| Integrated Gradients| Нет (градиенты)| Локальная          | ~0.1 сек (m=50)                | Да (аксиомы атрибуции)        | Да (быстро)     | Да |

*Примечания.* Время указано для одного объяснения на современном CPU без GPU; для Integrated Gradients с GPU время значительно меньше. Для KernelSHAP число сэмплов подмножеств обычно берут порядка $2D + 2048$.

#### 5.2. Какой метод выбрать? (шпаргалка)

Выбор метода определяется типом модели, требованиями к скорости, допустимостью доступа к градиентам и необходимостью глобальной интерпретации.

- **Модель — деревья или ансамбли (Random Forest, XGBoost, LightGBM):** TreeSHAP — безусловный лидер. Он даёт точные значения Шепли за полиномиальное время, поддерживает глобальную важность, summary и dependence plots. Если нужна только быстрая локальная аппроксимация без гарантий, можно использовать LIME, но TreeSHAP надёжнее.
- **Модель — нейронная сеть для изображений:** Integrated Gradients — оптимальный выбор. Он быстр, использует градиенты, даёт чёткие карты значимости. Альтернативы: Guided Backpropagation (менее обоснован) или LIME на суперпикселях (медленнее).
- **Модель — нейронная сеть для табличных данных:** можно применить Integrated Gradients (если архитектура дифференцируема) или KernelSHAP. IG быстрее, KernelSHAP даёт согласованные аддитивные вклады и глобальные метрики. При большом числе признаков ($D > 30$) KernelSHAP становится слишком медленным, и IG предпочтительнее.
- **Модель — чёрный ящик без градиентов (SVM с RBF-ядром, K‑ближайших соседей):** LIME — быстрый и простой вариант. KernelSHAP обеспечивает теоретические гарантии, но требует значительного времени. Для небольших $D$ (до 15) можно использовать точный SHAP.
- **Нужна глобальная интерпретация:** SHAP (TreeSHAP или KernelSHAP) предоставляет средние абсолютные значения (bar plot), summary plot и dependence plots. Альтернативы: permutation feature importance + Partial Dependence Plots (PDP) для любой модели, но они менее информативны о взаимодействиях.

#### 5.3. Оценка качества объяснений

Объяснение должно быть не только интуитивно понятным, но и объективно корректным. В отсутствие «золотого стандарта» применяют несколько эмпирических метрик.

1. **Визуальная оценка.** Для изображений тепловая карта атрибуций должна фокусироваться на объекте, а не на фоне. Для табличных данных waterfall plot должен логично отражать влияние признаков.

2. **Устойчивость (stability).** При малых возмущениях входа, не меняющих предсказание модели, объяснение не должно сильно меняться. Можно измерить, например, косинусное сходство между векторами атрибуций для $\mathbf{x}$ и $\mathbf{x} + \boldsymbol{\varepsilon}$.

3. **Верность (faithfulness).** Если признаки с наибольшими (по модулю) атрибуциями действительно важны, то их удаление или замена на базовые значения должно приводить к более сильному изменению предсказания, чем удаление признаков с малыми атрибуциями. Метрика faithfulness_score:

   ```python
   def faithfulness_score(model, X, shap_values, top_k=5):
       """
       Ожидаем shap_values для класса 1 (форма (n_samples, n_features)).
       Возвращает корреляцию между рангом важности и падением accuracy.
       """
       import numpy as np
       base_pred = model.predict(X)
       drops = []
       for i in range(len(X)):
           idx_sorted = np.argsort(-np.abs(shap_values[i]))
           top_features = idx_sorted[:top_k]
           X_pert = X[i].copy()
           X_pert[top_features] = np.median(X, axis=0)[top_features]  # замена на медианы
           new_pred = model.predict(X_pert.reshape(1, -1))[0]
           drops.append(base_pred[i] != new_pred)
       return np.mean(drops)
   ```

   Более высокая доля изменений предсказаний указывает на лучшее качество объяснения.

4. **Сравнение с известной истиной.** На синтетических данных, где истинные вклады признаков известны, можно вычислить коэффициент корреляции между истинными и оценёнными важностями.

#### 5.4. Сравнение на синтетических данных с известной истиной

Сгенерируем данные с аддитивной линейной зависимостью в логит-масштабе, чтобы истинные вклады были известны:

$$
\text{logit}(p) = 0.5 x_1 + 0.3 x_2 - 0.2 x_3 + \varepsilon, \quad x_j \sim \mathcal{N}(0,1),\; \varepsilon \sim \mathcal{N}(0,0.1).
$$

Класс $y = 1$ если $p > 0.5$. Обучим случайный лес и сравним оценки важности от TreeSHAP (среднее абсолютных SHAP), LIME и permutation importance.

```python
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import shap
import lime.lime_tabular

np.random.seed(42)
n = 2000
X = np.random.randn(n, 5)  # 5 признаков, два шумовых
logit = 0.5*X[:,0] + 0.3*X[:,1] - 0.2*X[:,2] + np.random.randn(n)*0.1
y = (1 / (1 + np.exp(-logit)) > 0.5).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

# TreeSHAP
explainer_shap = shap.TreeExplainer(rf)
shap_values = explainer_shap.shap_values(X_test)[1]
shap_importance = np.abs(shap_values).mean(axis=0)

# LIME
explainer_lime = lime.lime_tabular.LimeTabularExplainer(X_train, feature_names=['x1','x2','x3','x4','x5'],
                                                         class_names=['0','1'], mode='classification')
lime_importance = np.zeros(5)
for i in range(len(X_test)):
    exp = explainer_lime.explain_instance(X_test[i], rf.predict_proba, num_features=5)
    for feat, weight in exp.as_list():
        idx = int(feat.split('_')[0][1])  # извлечь индекс
        lime_importance[idx] += abs(weight)
lime_importance /= len(X_test)

# Permutation importance
from sklearn.inspection import permutation_importance
perm_imp = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm_importance_vals = perm_imp.importances_mean

print("Истинные коэффициенты: [0.5, 0.3, -0.2, 0, 0]")
print("SHAP importance:     ", np.round(shap_importance, 4))
print("LIME importance:     ", np.round(lime_importance, 4))
print("Permutation importance:", np.round(perm_importance_vals, 4))
```

Результаты (примерные): SHAP правильно выделяет x1, x2, x3; шумовые x4, x5 получают близкие к нулю значения. LIME также даёт верную картину, но с несколько большей вариацией. Permutation importance может переоценить важность коррелированных признаков, но здесь корреляций нет, и она тоже адекватна.

#### 5.5. Пример: объяснение одобрения кредита

Используем датасет `german credit` (или `credit` из sklearn, если доступен). Обучим XGBoost и с помощью TreeSHAP построим waterfall plots для двух клиентов — с одобренным и отклонённым кредитом.

```python
import xgboost as xgb
from sklearn.datasets import fetch_openml
import shap

# Загрузка German Credit Data
data = fetch_openml('credit-g', version=1, as_frame=True)
X = data.data
y = (data.target == 'good').astype(int)

# Кодирование категориальных признаков
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = xgb.XGBClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Waterfall для одобренного клиента
shap.waterfall_plot(shap.Explanation(values=shap_values[0], base_values=explainer.expected_value, data=X_test.iloc[0].values, feature_names=X.columns.tolist()))

# Waterfall для отклонённого
shap.waterfall_plot(shap.Explanation(values=shap_values[1], base_values=explainer.expected_value, data=X_test.iloc[1].values, feature_names=X.columns.tolist()))
```

На графиках видно, как признаки «толкают» предсказание от базового значения (среднее по выборке) к итоговой вероятности. Например, для одобренного клиента положительный вклад дают «срок кредита» (короткий) и «возраст» (старше), а отрицательный — «кредитная история» (наличие просрочек).

#### 5.6. Ограничения и предостережения

1. **Коррелированные признаки.** SHAP и LIME предполагают независимость признаков при маргинализации. При сильной корреляции вклады могут произвольно распределяться между взаимозаменяемыми признаками. Решения: группировка признаков, использование причинных SHAP.

2. **Причинность vs корреляция.** Объяснения описывают поведение модели, а не реального мира. Если модель выучила ложную корреляцию (например, наличие собаки на фото с травой), атрибуция укажет на траву как важный признак, хотя причиной метки является собака. Это не дефект интерпретации, а дефект модели.

3. **Объяснение не гарантирует корректность модели.** Можно получить красивое и непротиворечивое объяснение для модели, далёкой от истины. Интерпретация не заменяет валидацию и оценку точности.

4. **Проклятие размерности.** При $D > 100$ все методы локальной интерпретации страдают: LIME требует большого числа возмущений для покрытия пространства, KernelSHAP экспоненциально сложен, IG может давать шумные карты. Полезно предварительно снижать размерность или использовать встроенные методы важности для деревьев.

#### 5.7. Практический эксперимент: сравнение объяснений одной точки

Напишем функцию `compare_explainers`, которая для заданного объекта выводит топ-5 признаков по версии разных методов.

```python
def compare_explainers(model, X_train, X_test, idx, feature_names=None,
                       methods=['lime','shap','ig']):
    """
    Сравнивает объяснения одного объекта методами LIME, SHAP и IG.
    model: обученная модель (совместимая с sklearn API)
    X_train, X_test: numpy массивы
    idx: индекс тестового объекта
    feature_names: список названий признаков
    methods: список методов
    """
    x = X_test[idx]
    results = {}
    if feature_names is None:
        feature_names = [f"feat_{i}" for i in range(x.shape[0])]
    
    if 'lime' in methods:
        import lime.lime_tabular
        explainer = lime.lime_tabular.LimeTabularExplainer(
            X_train, feature_names=feature_names, mode='classification')
        exp = explainer.explain_instance(x, model.predict_proba, num_features=5)
        results['LIME'] = {feat: weight for feat, weight in exp.as_list()}
    
    if 'shap' in methods:
        import shap
        # Пытаемся TreeExplainer, если не выходит — KernelExplainer
        try:
            explainer = shap.TreeExplainer(model)
        except:
            explainer = shap.KernelExplainer(model.predict_proba, X_train[:100])
        shap_vals = explainer.shap_values(x.reshape(1,-1))
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]  # для бинарной классификации
        results['SHAP'] = {feature_names[i]: shap_vals[0][i] for i in range(len(x))}
    
    if 'ig' in methods:
        # Только для нейронных сетей; здесь заглушка
        pass
    
    # Вывод топ-5
    for method, weights in results.items():
        sorted_items = sorted(weights.items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
        print(f"{method}:")
        for feat, w in sorted_items:
            print(f"  {feat}: {w:.4f}")
        print()
```

Запустив эту функцию на тестовом объекте, можно визуально сравнить, насколько согласованы методы. Обычно TreeSHAP и KernelSHAP дают близкие результаты, LIME — сходную картину, но с возможными выбросами.

#### 5.8. Итоговые рекомендации

- **Для древовидных моделей** всегда предпочитайте TreeSHAP: он быстр, точен и даёт глобальную важность.
- **Для нейросетей** на изображениях и текстах используйте Integrated Gradients (с GPU) или его варианты. Для табличных данных — комбинируйте IG (быстрый скрининг) и KernelSHAP (финальные выводы).
- **Когда модель — полный чёрный ящик**, а время критично, начинайте с LIME. Для ответственных приложений переходите к KernelSHAP.
- **Не полагайтесь на одно объяснение.** Сравнивайте результаты нескольких методов; расхождения помогают обнаружить нестабильность или скрытые взаимодействия.

Интерпретируемость — не панацея, а инструмент. Правильно применённый, он превращает чёрный ящик в стеклянный, позволяя доверять модели, отлаживать её и извлекать новые знания о данных.

---

#### Вопросы для самопроверки

**Для начинающих:**

1. Как оценить, насколько правдоподобно объяснение, не имея эталонных вкладов признаков?
2. Какой метод интерпретации самый быстрый для ансамбля деревьев и почему?
3. Можно ли полностью доверять любому объяснению «чёрного ящика»? Аргументируйте.
4. Что делать, если два важных признака сильно коррелированы? Как это влияет на SHAP и LIME?
5. В каком случае стоит предпочесть LIME методу SHAP?

**Для профессионалов:**

1. Реализуйте метрику устойчивости (stability) объяснений: для случайного леса вычислите среднее косинусное сходство SHAP-значений при добавлении малого шума к входу.
2. Предложите метод агрегации объяснений от нескольких разных моделей (ансамбля интерпретаций). Как учесть согласованность?
3. Сравните SHAP и метод DeepLIFT для полносвязной нейронной сети. В чём различие в определении «базового» значения?
4. Как интерпретировать SHAP-значения, если модель возвращает вероятности, а не логиты? Нужно ли применять логит-преобразование?
5. Проведите анализ чувствительности Integrated Gradients к выбору базовой точки: вычислите IG для MNIST с нулевым, средним и размытым изображением в качестве baseline. Как меняются карты атрибуции?